<a href="https://colab.research.google.com/github/Madathanapalleleena/DL_exploration/blob/main/fashion_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fashion-MNIST**

28×28 grayscale images of clothing items

10 classes (T-shirt, Trouser, etc.)

~60k train, 10k test

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Datasets
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)  # 10 classes

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP().to(device)

In [10]:
def train_model(model, train_loader, optimizer, criterion, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(train_loader):.4f}")

def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc

In [11]:
criterion = nn.CrossEntropyLoss()
epochs = 5  # reduce for quick experiments

# Dictionary of optimizers
optimizers = {
    "BGD": optim.SGD(model.parameters(), lr=0.1),  # batch gradient descent (simulate by large batch)
    "SGD": optim.SGD(model.parameters(), lr=0.01),
    "MiniBatchGD": optim.SGD(model.parameters(), lr=0.01),  # batch_size < full dataset
    "SGD_Momentum": optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
    "SGD_Nesterov": optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True),
    "Adagrad": optim.Adagrad(model.parameters(), lr=0.01),
    "RMSProp": optim.RMSprop(model.parameters(), lr=0.01),
    "Adadelta": optim.Adadelta(model.parameters(), lr=1.0),
    "Adam": optim.Adam(model.parameters(), lr=0.001)
}

In [14]:
# Step 0: Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import copy

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Step 1: Load Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Step 2: Define a simple MLP
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

# Step 3: Training and testing functions
def train_model(model, train_loader, optimizer, criterion, epochs=5, verbose=True):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Print only for first batch if verbose is True
            if verbose and i == 0:
                print(f"Batch {i+1}: Loss={loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {avg_loss:.4f}")

def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc

# Step 4: Reset model function
def reset_model():
    return MLP().to(device)

# Step 5: Prepare optimizers with tuned learning rates
criterion = nn.CrossEntropyLoss()
epochs = 10  # enough for convergence

batch_sizes = {
    "BGD": len(train_dataset),  # Full batch
    "SGD": 1,                   # stochastic
    "MiniBatchGD": 64,           # mini batch
}

optimizer_configs = {
    "BGD": lambda params: optim.SGD(params, lr=0.1),
    "SGD": lambda params: optim.SGD(params, lr=0.001) ,
    "MiniBatchGD": lambda params: optim.SGD(params, lr=0.01),
    "SGD_Momentum": lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    "SGD_Nesterov": lambda params: optim.SGD(params, lr=0.01, momentum=0.9, nesterov=True),
    "Adagrad": lambda params: optim.Adagrad(params, lr=0.01),
    "RMSProp": lambda params: optim.RMSprop(params, lr=0.01),
    "Adadelta": lambda params: optim.Adadelta(params, lr=1.0),
    "Adam": lambda params: optim.Adam(params, lr=0.001),
}

# Step 6: Run experiments
results = {}

for name, opt_fn in optimizer_configs.items():
    print("\n" + "="*30)
    print(f"Training with {name}")

    # Set batch size
    bs = batch_sizes.get(name, 64)
    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # Reset model
    model = reset_model()
    optimizer = opt_fn(model.parameters())

    # Train
    train_model(model, train_loader, optimizer, criterion, epochs=epochs, verbose=False)

    # Test
    acc = test_model(model, test_loader)
    results[name] = acc

# Step 7: Summary
print("\n" + "="*30)
print("Summary of Test Accuracies:")
for k,v in results.items():
    print(f"{k}: {v:.2f}%")

Using device: cpu

Training with BGD
Epoch [1/10] Avg Loss: 2.2967
Epoch [2/10] Avg Loss: 2.2636
Epoch [3/10] Avg Loss: 2.2318
Epoch [4/10] Avg Loss: 2.1987
Epoch [5/10] Avg Loss: 2.1624
Epoch [6/10] Avg Loss: 2.1215
Epoch [7/10] Avg Loss: 2.0755
Epoch [8/10] Avg Loss: 2.0236
Epoch [9/10] Avg Loss: 1.9655
Epoch [10/10] Avg Loss: 1.9017
Test Accuracy: 47.87%

Training with SGD
Epoch [1/10] Avg Loss: 0.5838
Epoch [2/10] Avg Loss: 0.4029
Epoch [3/10] Avg Loss: 0.3620
Epoch [4/10] Avg Loss: 0.3354
Epoch [5/10] Avg Loss: 0.3134
Epoch [6/10] Avg Loss: 0.2969
Epoch [7/10] Avg Loss: 0.2834
Epoch [8/10] Avg Loss: 0.2703
Epoch [9/10] Avg Loss: 0.2596
Epoch [10/10] Avg Loss: 0.2495
Test Accuracy: 87.95%

Training with MiniBatchGD
Epoch [1/10] Avg Loss: 0.9812
Epoch [2/10] Avg Loss: 0.5462
Epoch [3/10] Avg Loss: 0.4826
Epoch [4/10] Avg Loss: 0.4498
Epoch [5/10] Avg Loss: 0.4267
Epoch [6/10] Avg Loss: 0.4089
Epoch [7/10] Avg Loss: 0.3960
Epoch [8/10] Avg Loss: 0.3832
Epoch [9/10] Avg Loss: 0.3720
E

Conclusions

Adaptive optimizers (Adam, RMSProp) usually outperform plain SGD on Fashion-MNIST for MLP.

Momentum & Nesterov accelerate convergence, better than vanilla SGD.

Batch size affects BGD/MiniBatchGD performance — full BGD can be slow.

For small networks & simple datasets, differences are visible but moderate.

Observation for regularization (if you add Dropout, L2, noise, etc.) → can further improve accuracy slightly and reduce overfitting.

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
import numpy as np
import copy
import random

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Data Augmentation

In [16]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Datasets
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

Dropout and Noise Option

Dropout layers included

Can optionally add Gaussian noise to inputs

Parameter sharing/tie is generally done for convolutional layers (we’ll do in CNN later)

In [17]:
class MLP_Reg(nn.Module):
    def __init__(self, input_size=28*28, hidden1=256, hidden2=128, dropout=0.5, noise_std=0.0):
        super(MLP_Reg, self).__init__()
        self.noise_std = noise_std
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std  # adding noise to input
        x = self.relu1(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu2(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

**Early Stopping**

Early stopping implemented using patience

Keeps best model weights

In [18]:
def train_model(model, train_loader, optimizer, criterion, epochs=20, patience=3, verbose=False):
    model.train()
    best_loss = float('inf')
    counter = 0
    best_model = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        if verbose:
            print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

        # Early stopping
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break
    model.load_state_dict(best_model)
    return model

def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    return acc

In [ ]:
# L2 regularization (weight decay)
weight_decays = [0, 0.001, 0.01]

# Dropout / noise
dropout_rates = [0, 0.3, 0.5]
noise_levels  = [0, 0.1]

results = []

for wd in weight_decays:
    for dr in dropout_rates:
        for noise in noise_levels:
            model = MLP_Reg(dropout=dr, noise_std=noise).to(device)
            optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=wd)
            model = train_model(model, train_loader, optimizer, nn.CrossEntropyLoss(), epochs=20, patience=5, verbose=False)
            acc = test_model(model, test_loader)
            results.append({
                "L2": wd,
                "Dropout": dr,
                "Noise": noise,
                "Test Accuracy": acc
            })

# Print summary
for r in results:
    print(r)

**Ensemble Method**

In [ ]:
ensemble_models = []
for i in range(3):
    model = MLP_Reg(dropout=0.5, noise_std=0.1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
    model = train_model(model, train_loader, optimizer, nn.CrossEntropyLoss(), epochs=20, patience=5, verbose=False)
    ensemble_models.append(model)

# Ensemble prediction
def ensemble_test(models, test_loader):
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = sum([torch.softmax(m(images), dim=1) for m in models]) / len(models)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    return acc

ensemble_acc = ensemble_test(ensemble_models, test_loader)
print(f"Ensemble Test Accuracy: {ensemble_acc:.2f}%")

Conclusions:
Dropout and L2 reduce overfitting

Input noise acts as data augmentation

Ensemble combines predictions → most stable & accurate

Early stopping prevents over-training

**CNN with Parameter Tuning**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, dropout=0.5):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

cnn_model = SimpleCNN(dropout=0.5).to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=0.001)
cnn_model = train_model(cnn_model, train_loader, optimizer, nn.CrossEntropyLoss(), epochs=20, patience=5, verbose=True)
cnn_acc = test_model(cnn_model, test_loader)
print(f"CNN Test Accuracy: {cnn_acc:.2f}%")